## Environment and Protocol Setup

In [4]:
# ============================================================
# Cell 01 - Environment & Imports
# ============================================================

# ------------------------------------------------------------
# Standard Library Imports
# ------------------------------------------------------------

import os
import sys
import time
import pickle
import json
import hashlib
import warnings
from pathlib import Path

# ------------------------------------------------------------
# Scientific Computing and Data Analysis
# ------------------------------------------------------------

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Machine Learning Libraries
# ------------------------------------------------------------

import sklearn
import xgboost as xgb

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)


# ------------------------------------------------------------
# Project Directory Path Resolution
# ------------------------------------------------------------
# Resolve the directory containing the shared project modules.
# An explicit environment variable takes priority; otherwise,
# the notebook searches common execution locations.

CODE_DIR_ENV = os.environ.get("THESIS_CODE_DIR")

if CODE_DIR_ENV:
    # Use the user-defined project directory when provided.
    CODE_DIR = Path(CODE_DIR_ENV).resolve()
else:
    candidates = []
    cwd = Path.cwd()

    # Check the current working directory for both required modules.
    if (
        (cwd / "data_pipeline.py").exists()
        and (cwd / "experiment_protocol.py").exists()
    ):
        candidates.append(cwd)

    # Check Kaggle input directories when running in a Kaggle environment.
    kaggle_input = Path("/kaggle/input")

    if kaggle_input.exists():
        for pipeline_file in kaggle_input.rglob("data_pipeline.py"):
            candidate_dir = pipeline_file.parent

            # A valid project directory must contain both shared modules.
            if (candidate_dir / "experiment_protocol.py").exists():
                candidates.append(candidate_dir)

    # Remove duplicate paths after resolving them to absolute paths.
    candidates = list(
        dict.fromkeys(path.resolve() for path in candidates)
    )

    if len(candidates) == 1:
        CODE_DIR = candidates[0]

    elif len(candidates) == 0:
        # Fail explicitly rather than importing from an unintended location.
        raise FileNotFoundError(
            "Could not locate data_pipeline.py and experiment_protocol.py. "
            "Set THESIS_CODE_DIR explicitly if necessary."
        )

    else:
        # Ambiguous project locations could cause inconsistent experiment code.
        raise RuntimeError(
            "Multiple valid code directories were found:\n\n"
            + "\n".join(str(path) for path in candidates)
            + "\n\nSet THESIS_CODE_DIR explicitly."
        )

# Make the shared project modules importable from the notebook.
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))


# ------------------------------------------------------------
# Central Project Protocol
# ------------------------------------------------------------
# Import the shared data-loading function and experiment constants.
# Keeping these settings in one protocol module ensures that the
# same experimental configuration is used across notebooks.

from data_pipeline import load_prepared_experiment

from experiment_protocol import (
    PROTOCOL_VERSION,
    EXPECTED_DATASETS,
    TRAINING_SIZES,
    SUBSET_SEEDS,
    TEST_SIZE,
    TEST_SEED,
    MODEL_SEED,
    CV_SEED,
    HPO_SEED,
    CLASSIFICATION_THRESHOLD,
    PRIMARY_METRICS,
    SECONDARY_METRICS,
    TRADITIONAL_MODELS,
    LOGISTIC_REGRESSION_MAX_ITER,
    XGBOOST_OBJECTIVE,
    XGBOOST_EVAL_METRIC,
    HPO_MIN_TRAINING_SIZE,
    HPO_TRAINING_SIZES,
    HPO_N_ITER,
    HPO_CV_SPLITS,
    HPO_SCORING,
)


# Combine the primary and secondary metric definitions for
# convenient access when evaluating experimental results.
ALL_METRICS = PRIMARY_METRICS + SECONDARY_METRICS

# Use all available CPU cores for hyperparameter search.
HPO_N_JOBS = -1

# Record the HPO implementation used by the experiment.
HPO_METHOD = "RandomizedSearchCV"


# Suppress a known non-critical FutureWarning emitted by
# scikit-learn's ColumnTransformer implementation.
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module=r"sklearn\.compose\._column_transformer",
)


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
# Set NumPy's global random seed for operations that rely on
# NumPy's global random-number generator.
np.random.seed(MODEL_SEED)


# ------------------------------------------------------------
# Environment and Protocol Summary
# ------------------------------------------------------------
# Print the execution environment and central protocol settings.
# This provides a compact audit trail for experiment reproducibility.

print("Traditional ML + HPO — revised protocol")
print("=" * 60)

print(f"Python:              {sys.version.split()[0]}")
print(f"NumPy:               {np.__version__}")
print(f"Pandas:              {pd.__version__}")
print(f"scikit-learn:        {sklearn.__version__}")
print(f"XGBoost:             {xgb.__version__}")

print(f"Protocol:            {PROTOCOL_VERSION}")
print(f"Project code dir:    {CODE_DIR}")
print(f"Datasets:            {EXPECTED_DATASETS}")
print(f"Training sizes:      {TRAINING_SIZES}")

# HPO-specific training sizes may differ from the main experiment sizes.
print(f"HPO sizes:           {HPO_TRAINING_SIZES}")

# Report all seeds used by the experimental protocol.
print(f"Subset seeds:        {SUBSET_SEEDS}")
print(f"Model seed:          {MODEL_SEED}")
print(f"CV seed:             {CV_SEED}")
print(f"HPO seed:            {HPO_SEED}")

# Report the computational configuration for hyperparameter optimization.
print(f"HPO iterations:      {HPO_N_ITER}")
print(f"HPO CV splits:       {HPO_CV_SPLITS}")
print(f"HPO scoring:         {HPO_SCORING}")
print(f"HPO jobs:             {HPO_N_JOBS}")

# Report the threshold used to convert predicted probabilities
# into binary class predictions during metric calculation.
print(f"Classification thr.: {CLASSIFICATION_THRESHOLD}")

print("=" * 60)
print("✓ Environment and central protocol loaded.")

Traditional ML + HPO — revised protocol
Python:              3.12.13
NumPy:               2.0.2
Pandas:              2.3.3
scikit-learn:        1.6.1
XGBoost:             3.2.0
Protocol:            low_data_tabpfn_v1
Project code dir:    /kaggle/input/datasets/nicolausdeemese/iu-thesis-nicolas/files
Datasets:            ('diabetes', 'cardiovascular')
Training sizes:      (8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192)
HPO sizes:           (32, 64, 128, 256, 512, 1024, 2048, 4096, 8192)
Subset seeds:        (42, 123, 456, 789, 1011)
Model seed:          42
CV seed:             42
HPO seed:            42
HPO iterations:      20
HPO CV splits:       5
HPO scoring:         roc_auc
HPO jobs:             -1
Classification thr.: 0.5
✓ Environment and central protocol loaded.


## Load Prepared Experimental Data

In [5]:
# ============================================================
# Cell 02 - Load Prepared Experimental Data
# ============================================================

print("Loading prepared experimental data...")
print("-" * 60)

# Load the prepared datasets and their associated metadata through
# the centralized data pipeline. This ensures that preprocessing
# and dataset preparation follow the same procedure across experiments.
experiment_data = load_prepared_experiment()

print("✓ Prepared experimental data loaded successfully.")

# Display the top-level components returned by the data pipeline.
# These keys identify the datasets, metadata, and other objects
# available to the subsequent experimental cells.
print(f"Prepared data keys: {list(experiment_data.keys())}")

# The conditional expression intentionally prevents access to the
# dataset path here. The actual data location is managed internally
# by the preparation pipeline, so only the pipeline-level source is
# reported in this cell.
print(
    f"Prepared data directory: "
    f"{Path(experiment_data['manifest'].get('datasets', {})
        .get(EXPECTED_DATASETS[0], {})
        .get('cleaned_file', '')).parent
      if False else 'loaded from pipeline'}"
)

Loading prepared experimental data...
------------------------------------------------------------
✓ Prepared experimental data loaded successfully.
Prepared data keys: ['prepared_data', 'split_data', 'training_subsets', 'config', 'manifest']
Prepared data directory: loaded from pipeline


## Verify Experimental Data Structure

In [6]:
# ============================================================
# Cell 03 - Extract and Verify Experimental Data Components
# ============================================================

# Define the exact top-level structure expected from the data pipeline.
# This prevents silent failures caused by missing or unexpected components.
EXPECTED_KEYS = {
    "training_subsets",
    "split_data",
    "prepared_data",
    "config",
    "manifest",
}

# Verify that the pipeline returned exactly the expected components.
assert set(experiment_data.keys()) == EXPECTED_KEYS, (
    f"Unexpected experiment data structure: {set(experiment_data.keys())}"
)

# Extract the main experimental components for direct use in later cells.
training_subsets = experiment_data["training_subsets"]
split_data = experiment_data["split_data"]
prepared_data = experiment_data["prepared_data"]
config = experiment_data["config"]
manifest = experiment_data["manifest"]


# ------------------------------------------------------------
# Protocol ↔ Prepared-Data Consistency
# ------------------------------------------------------------
# Verify that the preparation settings stored with the data match
# the centralized experimental protocol loaded in Cell 01.
#
# These checks are important because the prepared datasets may have
# been generated separately from the current notebook session.

assert tuple(config["training_sizes"]) == tuple(TRAINING_SIZES)
assert tuple(config["training_seeds"]) == tuple(SUBSET_SEEDS)

assert config["test_size"] == TEST_SIZE
assert config["test_seed"] == TEST_SEED

# The experiment is configured as binary classification.
assert config["n_classes"] == 2


# ------------------------------------------------------------
# Dataset Consistency
# ------------------------------------------------------------
# Confirm that all major data components contain exactly the datasets
# specified by the central experimental protocol.

assert tuple(training_subsets.keys()) == EXPECTED_DATASETS
assert tuple(split_data.keys()) == EXPECTED_DATASETS
assert tuple(prepared_data.keys()) == EXPECTED_DATASETS


# ------------------------------------------------------------
# Verification Summary
# ------------------------------------------------------------

print("✓ Experiment structure verified.")
print("✓ Protocol matches prepared experiment manifest.")
print(f"✓ Datasets: {EXPECTED_DATASETS}")
print(f"✓ Training sizes: {TRAINING_SIZES}")
print(f"✓ Subset seeds: {SUBSET_SEEDS}")

✓ Experiment structure verified.
✓ Protocol matches prepared experiment manifest.
✓ Datasets: ('diabetes', 'cardiovascular')
✓ Training sizes: (8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192)
✓ Subset seeds: (42, 123, 456, 789, 1011)


## Verify Training and Test Sets

In [7]:
# ============================================================
# Cell 04 - Verify Training Subsets and Fixed Test Sets
# ============================================================

print("Verifying training subsets and fixed test sets...")
print("=" * 60)

# Validate each dataset independently to ensure that the fixed test set
# and all predefined training subsets satisfy the experimental protocol.
for dataset_name in EXPECTED_DATASETS:

    # Retrieve the fixed test set and the original train/test indices.
    X_test = split_data[dataset_name]["X_test"]
    y_test = split_data[dataset_name]["y_test"]

    test_indices = set(split_data[dataset_name]["test_indices"])
    train_indices = set(split_data[dataset_name]["train_indices"])

    # ------------------------------------------------------------
    # Fixed Test Set Integrity
    # ------------------------------------------------------------

    # The original train and test partitions must not overlap.
    assert train_indices.isdisjoint(test_indices)

    # Features and labels must contain the same number of observations.
    assert len(X_test) == len(y_test)

    # Prepared test features and labels must contain no missing values.
    assert not X_test.isnull().any().any()
    assert not y_test.isnull().any()

    # Confirm binary classification labels {0, 1}.
    assert set(np.asarray(y_test)) == {0, 1}

    print()
    print(f"Dataset: {dataset_name}")
    print(f"  Test rows:     {len(X_test):,}")
    print(f"  Test features: {X_test.shape[1]}")


    # ------------------------------------------------------------
    # Training Subset Integrity
    # ------------------------------------------------------------
    # Each configured training size and subset seed must be present.
    # These subsets are generated from the training partition only.

    for training_size in TRAINING_SIZES:
        assert training_size in training_subsets[dataset_name]

        for subset_seed in SUBSET_SEEDS:
            assert subset_seed in training_subsets[dataset_name][training_size]

            subset = training_subsets[dataset_name][training_size][subset_seed]

            X_train = subset["X"]
            y_train = subset["y"]
            train_subset_indices = set(subset["indices"])

            # The subset must contain exactly the requested number of rows.
            assert len(X_train) == training_size
            assert len(y_train) == training_size

            # Training and test sets must use the same feature space.
            assert X_train.shape[1] == X_test.shape[1]

            # Prepared training data must contain no missing values.
            assert not X_train.isnull().any().any()
            assert not y_train.isnull().any()

            # Confirm binary classification labels {0, 1}.
            assert set(np.asarray(y_train)) == {0, 1}

            # The stored indices must correspond one-to-one with the subset rows.
            assert len(train_subset_indices) == training_size

            # Every subset observation must originate from the fixed
            # training partition and must never appear in the test set.
            assert train_subset_indices.issubset(train_indices)
            assert train_subset_indices.isdisjoint(test_indices)

            # Each training subset is required to be perfectly balanced:
            # half class 0 and half class 1.
            counts = y_train.value_counts().sort_index()

            assert counts.to_dict() == {
                0: training_size // 2,
                1: training_size // 2,
            }


    # ------------------------------------------------------------
    # Dataset-Level Verification Summary
    # ------------------------------------------------------------

    print("  ✓ Fixed test set verified.")
    print(f"  ✓ {len(TRAINING_SIZES)} training sizes verified.")
    print(f"  ✓ {len(SUBSET_SEEDS)} balanced subsets per size verified.")


# ------------------------------------------------------------
# Global Verification Summary
# ------------------------------------------------------------

print()
print("=" * 60)
print("✓ ALL DATA INTEGRITY CHECKS PASSED")
print("=" * 60)

Verifying training subsets and fixed test sets...

Dataset: diabetes
  Test rows:     45,895
  Test features: 21
  ✓ Fixed test set verified.
  ✓ 11 training sizes verified.
  ✓ 5 balanced subsets per size verified.

Dataset: cardiovascular
  Test rows:     13,723
  Test features: 11
  ✓ Fixed test set verified.
  ✓ 11 training sizes verified.
  ✓ 5 balanced subsets per size verified.

✓ ALL DATA INTEGRITY CHECKS PASSED


## Configure HPO Protocol

In [8]:
# ============================================================
# Cell 05 - HPO Protocol Configuration
# ============================================================

print("HPO protocol configuration")
print("=" * 60)


# ------------------------------------------------------------
# Validate Central HPO Protocol
# ------------------------------------------------------------
# Confirm that the HPO settings loaded from the central protocol
# satisfy the requirements of the experimental design.

assert HPO_METHOD == "RandomizedSearchCV"

# HPO begins only at or above the minimum training size defined
# by the experimental protocol.
assert HPO_MIN_TRAINING_SIZE in TRAINING_SIZES

# Verify that the HPO training sizes are exactly the configured
# training sizes meeting the minimum-size criterion.
assert HPO_TRAINING_SIZES == tuple(
    size for size in TRAINING_SIZES
    if size >= HPO_MIN_TRAINING_SIZE
)

# At least one hyperparameter configuration must be evaluated.
assert HPO_N_ITER >= 1

# Stratified K-fold CV requires at least two folds.
assert HPO_CV_SPLITS >= 2

# ROC AUC is the fixed optimization metric for hyperparameter search.
assert HPO_SCORING == "roc_auc"


# ------------------------------------------------------------
# Separate Default-Training and HPO Sizes
# ------------------------------------------------------------
# Smaller training sizes are evaluated without HPO when they fall
# below the predefined HPO threshold. This keeps computational cost
# controlled while preserving the complete training-size experiment.

DEFAULT_TRAINING_SIZES = tuple(
    size for size in TRAINING_SIZES
    if size < HPO_MIN_TRAINING_SIZE
)


# ------------------------------------------------------------
# Cross-Validation Strategy
# ------------------------------------------------------------
# Create a deterministic stratified K-fold strategy for HPO.
# Stratification preserves the class distribution across folds,
# while shuffling reduces dependence on the original row ordering.

def create_cv_strategy(training_size):
    # Prevent accidental use of this CV strategy outside the sizes
    # explicitly designated for hyperparameter optimization.
    assert training_size in HPO_TRAINING_SIZES

    return StratifiedKFold(
        n_splits=HPO_CV_SPLITS,
        shuffle=True,
        random_state=CV_SEED,
    )


# ------------------------------------------------------------
# HPO Configuration Summary
# ------------------------------------------------------------
# Print the effective configuration for experiment traceability.

print(f"Search method:       {HPO_METHOD}")
print(f"Scoring:              {HPO_SCORING}")
print(f"Iterations:           {HPO_N_ITER}")
print(f"CV splits:            {HPO_CV_SPLITS}")
print(f"Parallel jobs:        {HPO_N_JOBS}")
print(f"HPO sizes:            {HPO_TRAINING_SIZES}")
print(f"Excluded from HPO:    {DEFAULT_TRAINING_SIZES}")
print(f"CV seed:              {CV_SEED}")

print()
print("✓ HPO configuration verified against central protocol.")

HPO protocol configuration
Search method:       RandomizedSearchCV
Scoring:              roc_auc
Iterations:           20
CV splits:            5
Parallel jobs:        -1
HPO sizes:            (32, 64, 128, 256, 512, 1024, 2048, 4096, 8192)
Excluded from HPO:    (8, 16)
CV seed:              42

✓ HPO configuration verified against central protocol.


## Define HPO Models and Search Spaces

In [9]:
# ============================================================
# Cell 06 - Define HPO Models and Search Spaces
# ============================================================

print("Defining HPO models and search spaces...")
print("=" * 60)


# ------------------------------------------------------------
# Logistic Regression
# ------------------------------------------------------------
# Standardize features before fitting Logistic Regression because
# its optimization is sensitive to differences in feature scale.
# The classifier is wrapped in a Pipeline so scaling is performed
# independently within each CV fold, preventing data leakage.

logistic_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                max_iter=LOGISTIC_REGRESSION_MAX_ITER,
                random_state=MODEL_SEED,
            ),
        ),
    ]
)

# Search space for Logistic Regression.
# C controls inverse regularization strength: smaller values imply
# stronger regularization. Only the L2 penalty with the LBFGS solver
# is considered in accordance with the experimental protocol.
logistic_search_space = {
    "classifier__C": [
        0.001,
        0.003,
        0.01,
        0.03,
        0.1,
        0.3,
        1.0,
        3.0,
        10.0,
        30.0,
    ],
    "classifier__penalty": ["l2"],
    "classifier__solver": ["lbfgs"],
    "classifier__class_weight": [None, "balanced"],
}


# ------------------------------------------------------------
# Random Forest
# ------------------------------------------------------------

# Random Forest does not require feature standardization.
# The random_state fixes the model's stochastic behavior.
random_forest_model = RandomForestClassifier(
    random_state=MODEL_SEED,
)

# Search space covering tree complexity, ensemble size,
# minimum leaf size, and feature-selection strategy.
random_forest_search_space = {
    "n_estimators": [100, 300],
    "max_depth": [5, 10, None],
    "min_samples_leaf": [1, 5],
    "max_features": ["sqrt", "log2", None],
}


# ------------------------------------------------------------
# XGBoost
# ------------------------------------------------------------

# Configure XGBoost using the objective and evaluation metric
# defined by the central experimental protocol.
xgboost_model = xgb.XGBClassifier(
    objective=XGBOOST_OBJECTIVE,
    eval_metric=XGBOOST_EVAL_METRIC,
    random_state=MODEL_SEED,
)

# Search space for boosting depth, learning rate, ensemble size,
# row subsampling, and feature subsampling.
xgboost_search_space = {
    "max_depth": [3, 6, 9],
    "learning_rate": [0.01, 0.1, 0.3],
    "n_estimators": [100, 300],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}


# ------------------------------------------------------------
# Unified HPO Model Registry
# ------------------------------------------------------------
# Store each model together with its corresponding search space.
# This allows the HPO execution code to process all models through
# the same generic workflow.

HPO_MODELS = {
    "logistic_regression": (
        logistic_model,
        logistic_search_space,
    ),
    "random_forest": (
        random_forest_model,
        random_forest_search_space,
    ),
    "xgboost": (
        xgboost_model,
        xgboost_search_space,
    ),
}

MODEL_NAMES = tuple(HPO_MODELS.keys())

# Ensure that the models used for HPO exactly match the models
# specified by the central experimental protocol.
assert MODEL_NAMES == TRADITIONAL_MODELS


# ------------------------------------------------------------
# Verify Search-Space Cardinalities
# ------------------------------------------------------------
# Calculate the total number of possible hyperparameter combinations.
# RandomizedSearchCV samples from these combinations rather than
# evaluating the full Cartesian product.

def search_space_cardinality(search_space):
    total = 1

    for values in search_space.values():
        total *= len(values)

    return total


space_sizes = {
    name: search_space_cardinality(space)
    for name, (_, space) in HPO_MODELS.items()
}

# Each search space must contain at least as many configurations
# as the number of randomized iterations requested.
assert space_sizes["logistic_regression"] >= HPO_N_ITER
assert space_sizes["random_forest"] >= HPO_N_ITER
assert space_sizes["xgboost"] >= HPO_N_ITER


# ------------------------------------------------------------
# Configuration Summary
# ------------------------------------------------------------

print("Models:")

for model_name in MODEL_NAMES:
    print(
        f"  ✓ {model_name}: "
        f"{space_sizes[model_name]} possible configurations"
    )

print()
print("Execution settings:")
print(f"  ✓ Logistic Regression max_iter={LOGISTIC_REGRESSION_MAX_ITER}")
print(f"  ✓ XGBoost objective={XGBOOST_OBJECTIVE}")
print(f"  ✓ XGBoost eval_metric={XGBOOST_EVAL_METRIC}")
print(f"  ✓ Model seed={MODEL_SEED}")

print()
print("✓ HPO model definitions and search spaces ready.")

Defining HPO models and search spaces...
Models:
  ✓ logistic_regression: 20 possible configurations
  ✓ random_forest: 36 possible configurations
  ✓ xgboost: 72 possible configurations

Execution settings:
  ✓ Logistic Regression max_iter=1000
  ✓ XGBoost objective=binary:logistic
  ✓ XGBoost eval_metric=logloss
  ✓ Model seed=42

✓ HPO model definitions and search spaces ready.


## Define Evaluation Function

In [10]:
# ============================================================
# Cell 07 - Evaluation Function
# ============================================================

print("Initializing evaluation function...")
print("=" * 60)


# ------------------------------------------------------------
# Prediction Evaluation
# ------------------------------------------------------------
# Convert predicted probabilities into the standardized set of
# evaluation metrics defined by the experimental protocol.
#
# Probability-based metrics:
#   - ROC AUC
#   - PR AUC
#
# Threshold-based metrics:
#   - Accuracy
#   - Balanced accuracy
#   - Precision
#   - Recall
#   - F1
#
# The classification threshold is fixed by the central protocol
# so that all models are evaluated under the same decision rule.

def evaluate_predictions(y_true, y_proba):

    # Convert inputs to NumPy arrays for consistent numerical handling.
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)

    # Predictions and probabilities must correspond one-to-one.
    assert len(y_true) == len(y_proba)

    # Reject NaN, infinite, or otherwise non-finite predicted probabilities.
    assert np.isfinite(y_proba).all()

    # Convert probabilities into binary predictions using the fixed
    # protocol threshold. This threshold is applied only to metrics
    # that require discrete class predictions.
    y_pred = (
        y_proba >= CLASSIFICATION_THRESHOLD
    ).astype(int)

    return {
        # ROC AUC evaluates ranking quality across all possible
        # classification thresholds.
        "roc_auc": float(
            roc_auc_score(y_true, y_proba)
        ),

        # Average precision summarizes the precision-recall curve
        # and is therefore based on predicted probabilities.
        "pr_auc": float(
            average_precision_score(y_true, y_proba)
        ),

        # The remaining metrics use the fixed binary predictions.
        "accuracy": float(
            accuracy_score(y_true, y_pred)
        ),

        "balanced_accuracy": float(
            balanced_accuracy_score(y_true, y_pred)
        ),

        # zero_division=0 prevents undefined precision/recall/F1
        # calculations from interrupting an experiment when a class
        # receives no predicted positive samples.
        "precision": float(
            precision_score(y_true, y_pred, zero_division=0)
        ),

        "recall": float(
            recall_score(y_true, y_pred, zero_division=0)
        ),

        "f1": float(
            f1_score(y_true, y_pred, zero_division=0)
        ),
    }


# ------------------------------------------------------------
# Evaluation Configuration Summary
# ------------------------------------------------------------

print(f"Primary metrics:   {PRIMARY_METRICS}")
print(f"Secondary metrics: {SECONDARY_METRICS}")
print(f"Threshold:          {CLASSIFICATION_THRESHOLD}")

print("✓ Evaluation function ready.")

Initializing evaluation function...
Primary metrics:   ('roc_auc', 'pr_auc')
Secondary metrics: ('accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1')
Threshold:          0.5
✓ Evaluation function ready.


## Define Single HPO Experiment

In [11]:
# ============================================================
# Cell 08 - Single HPO Experiment Function
# ============================================================

def run_hpo_experiment(
    dataset_name,
    model_name,
    training_size,
    subset_seed,
):
    """
    Run one HPO experiment.

    Training data:
        One predefined balanced subset selected by subset_seed.

    Model selection:
        RandomizedSearchCV on the training subset only.
        CV uses CV_SEED.
        Parameter sampling uses HPO_SEED.
        Model estimators use MODEL_SEED.

    Final evaluation:
        The complete fixed external test set.
    """

    # --------------------------------------------------------
    # Validate experiment parameters
    # --------------------------------------------------------
    # Fail early if the requested configuration is not part of
    # the centralized experimental protocol.

    if dataset_name not in EXPECTED_DATASETS:
        raise ValueError(f"Unknown dataset: {dataset_name}")

    if model_name not in MODEL_NAMES:
        raise ValueError(f"Unknown model: {model_name}")

    if training_size not in HPO_TRAINING_SIZES:
        raise ValueError(
            f"HPO is not defined for training size {training_size}."
        )

    if subset_seed not in SUBSET_SEEDS:
        raise ValueError(f"Unknown subset seed: {subset_seed}")


    # --------------------------------------------------------
    # Retrieve predefined data
    # --------------------------------------------------------
    # Select one fixed balanced training subset using the dataset,
    # training size, and subset seed. The test set is always the
    # complete fixed external test partition.

    subset = training_subsets[
        dataset_name
    ][training_size][subset_seed]

    X_train = subset["X"]
    y_train = subset["y"]

    X_test = split_data[dataset_name]["X_test"]
    y_test = split_data[dataset_name]["y_test"]

    # Confirm that the selected training subset and external test
    # set have the expected dimensions.
    assert len(X_train) == training_size
    assert len(X_test) == len(y_test)


    # --------------------------------------------------------
    # Retrieve model and search space
    # --------------------------------------------------------
    # Each HPO model is stored together with its corresponding
    # hyperparameter search space in HPO_MODELS.

    base_model, search_space = HPO_MODELS[model_name]

    # Create a deterministic stratified CV strategy for this run.
    # The CV folds are generated only from the training subset.
    cv_strategy = create_cv_strategy(training_size)


    # --------------------------------------------------------
    # Hyperparameter Optimization
    # --------------------------------------------------------
    # RandomizedSearchCV samples HPO_N_ITER parameter combinations,
    # evaluates them using stratified CV, and refits the best
    # configuration on the complete training subset.
    #
    # Important separation of random states:
    #   CV_SEED    -> controls cross-validation fold generation
    #   HPO_SEED   -> controls hyperparameter sampling
    #   MODEL_SEED -> controls stochastic model behavior

    hpo_start = time.time()

    search = RandomizedSearchCV(
        estimator=base_model,
        param_distributions=search_space,
        n_iter=HPO_N_ITER,
        scoring=HPO_SCORING,
        cv=cv_strategy,
        random_state=HPO_SEED,
        n_jobs=HPO_N_JOBS,
        refit=True,
        return_train_score=False,
    )

    # HPO is performed exclusively on the predefined training subset.
    # The fixed external test set is not exposed during model selection.
    search.fit(X_train, y_train)

    hpo_time = time.time() - hpo_start


    # --------------------------------------------------------
    # Final Prediction on Fixed Test Set
    # --------------------------------------------------------
    # The refitted best estimator is applied once to the complete
    # external test set after hyperparameter selection is finished.

    prediction_start = time.time()

    y_proba_all = search.best_estimator_.predict_proba(
        X_test
    )

    prediction_time = time.time() - prediction_start

    # Retrieve the class ordering used by the fitted estimator.
    # Selecting the probability associated explicitly with class 1
    # avoids relying on an assumed column position.
    classes = np.asarray(search.best_estimator_.classes_)

    assert len(classes) == 2

    positive_indices = np.where(classes == 1)[0]

    # Exactly one probability column must correspond to the
    # positive class in this binary classification experiment.
    assert len(positive_indices) == 1

    y_proba = np.asarray(
        y_proba_all[:, positive_indices[0]]
    )

    assert len(y_proba) == len(y_test)


    # --------------------------------------------------------
    # External Test Evaluation
    # --------------------------------------------------------
    # Evaluate predictions against the fixed test labels using
    # the standardized evaluation function defined in Cell 07.

    evaluation = evaluate_predictions(
        y_true=y_test,
        y_proba=y_proba,
    )


    # --------------------------------------------------------
    # Return Experiment Result
    # --------------------------------------------------------
    # Store the protocol identifiers, experimental conditions,
    # selected hyperparameters, timing information, CV score,
    # and external test metrics in one serializable dictionary.

    return {
        "protocol_version": PROTOCOL_VERSION,
        "dataset": dataset_name,
        "model": model_name,
        "training_size": training_size,
        "subset_seed": subset_seed,

        # Random seeds are recorded to make the run configuration
        # explicitly traceable and reproducible.
        "model_seed": MODEL_SEED,
        "cv_seed": CV_SEED,
        "hpo_seed": HPO_SEED,

        "classification_threshold": CLASSIFICATION_THRESHOLD,

        # best_cv_score is the mean CV score of the selected
        # hyperparameter configuration, not the external test score.
        "best_cv_score": float(search.best_score_),

        # Convert the selected parameters to a regular dictionary
        # for convenient serialization and later result analysis.
        "best_params": dict(search.best_params_),

        "hpo_time": float(hpo_time),
        "prediction_time": float(prediction_time),

        # Append the standardized external-test metrics.
        **evaluation,
    }


print("✓ Single HPO experiment function defined.")

✓ Single HPO experiment function defined.


## HPO Smoke Test

In [12]:
# ============================================================
# Cell 09 - HPO Smoke Test
# ============================================================

# Use a small, representative configuration to verify that the
# complete HPO pipeline executes correctly before running the
# full experimental grid.
SMOKE_DATASET = EXPECTED_DATASETS[0]
SMOKE_MODEL = "random_forest"
SMOKE_TRAINING_SIZE = 128
SMOKE_SUBSET_SEED = SUBSET_SEEDS[0]


# ------------------------------------------------------------
# Smoke Test Configuration
# ------------------------------------------------------------

print("Running HPO smoke test...")
print("=" * 60)

print(f"Dataset:         {SMOKE_DATASET}")
print(f"Model:           {SMOKE_MODEL}")
print(f"Training size:   {SMOKE_TRAINING_SIZE}")
print(f"Subset seed:     {SMOKE_SUBSET_SEED}")
print(f"Model seed:      {MODEL_SEED}")
print(f"CV seed:         {CV_SEED}")
print(f"HPO seed:        {HPO_SEED}")

print("=" * 60)


# ------------------------------------------------------------
# Execute Single HPO Run
# ------------------------------------------------------------

# Measure the total wall-clock time of the smoke test, including
# HPO, model refitting, prediction, and evaluation.
smoke_start = time.time()

smoke_result = run_hpo_experiment(
    dataset_name=SMOKE_DATASET,
    model_name=SMOKE_MODEL,
    training_size=SMOKE_TRAINING_SIZE,
    subset_seed=SMOKE_SUBSET_SEED,
)

smoke_elapsed = time.time() - smoke_start


# ------------------------------------------------------------
# Display Smoke Test Results
# ------------------------------------------------------------

print()
print("Smoke test completed.")
print("-" * 60)

# Display all protocol-defined evaluation metrics using a
# consistent four-decimal format.
for metric in PRIMARY_METRICS + SECONDARY_METRICS:
    print(f"{metric:20s}: {smoke_result[metric]:.4f}")

print()

# best_cv_score is the mean cross-validation score of the selected
# hyperparameter configuration on the training subset.
print(f"Best CV score:    {smoke_result['best_cv_score']:.4f}")

# HPO time measures hyperparameter search and refitting.
print(f"HPO time:         {smoke_result['hpo_time']:.2f} s")

# Prediction time measures inference on the complete fixed test set.
print(f"Prediction time:  {smoke_result['prediction_time']:.2f} s")

# Total time includes the complete execution of this smoke test.
print(f"Total time:       {smoke_elapsed:.2f} s")


# ------------------------------------------------------------
# Selected Hyperparameters
# ------------------------------------------------------------

print()
print("Best parameters:")

for key, value in smoke_result["best_params"].items():
    print(f"  {key}: {value}")


# ------------------------------------------------------------
# Smoke Validation
# ------------------------------------------------------------
# Verify that the returned result corresponds exactly to the
# requested experimental configuration and central protocol.

assert smoke_result["protocol_version"] == PROTOCOL_VERSION
assert smoke_result["dataset"] == SMOKE_DATASET
assert smoke_result["model"] == SMOKE_MODEL
assert smoke_result["training_size"] == SMOKE_TRAINING_SIZE
assert smoke_result["subset_seed"] == SMOKE_SUBSET_SEED

# Confirm that all random-state settings were propagated correctly.
assert smoke_result["model_seed"] == MODEL_SEED
assert smoke_result["cv_seed"] == CV_SEED
assert smoke_result["hpo_seed"] == HPO_SEED

# Confirm that the evaluation threshold matches the central protocol.
assert (
    smoke_result["classification_threshold"]
    == CLASSIFICATION_THRESHOLD
)


# ------------------------------------------------------------
# Metric Validation
# ------------------------------------------------------------
# All configured classification metrics should be finite and lie
# within their expected [0, 1] range.

for metric in ALL_METRICS:
    value = smoke_result[metric]

    assert np.isfinite(value)
    assert 0.0 <= value <= 1.0


# The best CV ROC AUC score is also expected to be finite and
# bounded between 0 and 1.
assert np.isfinite(smoke_result["best_cv_score"])
assert 0.0 <= smoke_result["best_cv_score"] <= 1.0


# Runtime values must be non-negative.
assert smoke_result["hpo_time"] >= 0
assert smoke_result["prediction_time"] >= 0


# ------------------------------------------------------------
# Smoke Test Summary
# ------------------------------------------------------------

print()
print("=" * 60)
print("✓ HPO SMOKE TEST PASSED")
print("=" * 60)

Running HPO smoke test...
Dataset:         diabetes
Model:           random_forest
Training size:   128
Subset seed:     42
Model seed:      42
CV seed:         42
HPO seed:        42

Smoke test completed.
------------------------------------------------------------
roc_auc             : 0.7663
pr_auc              : 0.3209
accuracy            : 0.6437
balanced_accuracy   : 0.7097
precision           : 0.2738
recall              : 0.8048
f1                  : 0.4086

Best CV score:    0.8160
HPO time:         16.58 s
Prediction time:  0.59 s
Total time:       17.21 s

Best parameters:
  n_estimators: 300
  min_samples_leaf: 1
  max_features: sqrt
  max_depth: 10

✓ HPO SMOKE TEST PASSED


## Initialize Checkpoint and Resume State

In [13]:
# ============================================================
# Cell 10 - Initialize Checkpoint / Resume State
# ============================================================

# Store the checkpoint in Kaggle's persistent working directory
# so completed HPO runs can be recovered after interruptions.
CHECKPOINT_PATH = Path(
    "/kaggle/working/"
    "traditional_ml_hpo_revised_protocol_checkpoint.pkl"
)


# ------------------------------------------------------------
# Expected Experiment Size
# ------------------------------------------------------------
# Each run is uniquely defined by:
#   dataset × model × training size × subset seed
#
# This value is used later to verify that the complete experiment
# grid has been executed.

EXPECTED_RUNS = (
    len(EXPECTED_DATASETS)
    * len(HPO_TRAINING_SIZES)
    * len(SUBSET_SEEDS)
    * len(MODEL_NAMES)
)


# ------------------------------------------------------------
# Search-Space Signature
# ------------------------------------------------------------
# Hash the complete HPO search-space definition so that a checkpoint
# cannot be silently reused after the search space has changed.
#
# sort_keys=True makes the serialized representation deterministic.
# default=str allows non-JSON-native parameter values to be serialized.

SEARCH_SPACE_SIGNATURE = hashlib.sha256(
    json.dumps(
        {
            name: space
            for name, (_, space) in HPO_MODELS.items()
        },
        sort_keys=True,
        default=str,
    ).encode("utf-8")
).hexdigest()


# ------------------------------------------------------------
# Checkpoint Metadata
# ------------------------------------------------------------
# Store all protocol settings that materially affect the HPO results.
# The metadata must match exactly when an existing checkpoint is loaded.

CHECKPOINT_METADATA = {
    "protocol_version": PROTOCOL_VERSION,
    "hpo_method": HPO_METHOD,
    "hpo_training_sizes": tuple(HPO_TRAINING_SIZES),
    "subset_seeds": tuple(SUBSET_SEEDS),
    "model_names": tuple(MODEL_NAMES),
    "model_seed": MODEL_SEED,
    "cv_seed": CV_SEED,
    "hpo_seed": HPO_SEED,
    "hpo_n_iter": HPO_N_ITER,
    "hpo_cv_splits": HPO_CV_SPLITS,
    "hpo_scoring": HPO_SCORING,
    "classification_threshold": CLASSIFICATION_THRESHOLD,
    "search_space_signature": SEARCH_SPACE_SIGNATURE,
}


# ------------------------------------------------------------
# Save Checkpoint
# ------------------------------------------------------------

def save_checkpoint(results):
    """
    Save completed experiment results together with protocol metadata.

    A temporary file is written first and then atomically renamed to
    the target path. This reduces the risk of leaving a partially
    written checkpoint if the save operation is interrupted.
    """

    payload = {
        "metadata": CHECKPOINT_METADATA,
        "results": results,
    }

    # Use a temporary file in the same directory as the final
    # checkpoint so that the replacement operation remains local.
    temporary_path = CHECKPOINT_PATH.with_suffix(".tmp")

    with open(temporary_path, "wb") as handle:
        pickle.dump(
            payload,
            handle,
            protocol=pickle.HIGHEST_PROTOCOL,
        )

    # Replace the previous checkpoint only after serialization succeeds.
    temporary_path.replace(CHECKPOINT_PATH)


# ------------------------------------------------------------
# Load Checkpoint
# ------------------------------------------------------------

def load_checkpoint():
    """
    Load and validate an existing experiment checkpoint.

    The checkpoint is accepted only when its metadata exactly matches
    the current experimental protocol.
    """

    with open(CHECKPOINT_PATH, "rb") as handle:
        payload = pickle.load(handle)

    # The checkpoint must contain the expected dictionary structure.
    assert isinstance(payload, dict)

    # Exact metadata equality prevents resuming an experiment with
    # different models, seeds, search spaces, or HPO settings.
    assert payload["metadata"] == CHECKPOINT_METADATA, (
        "Checkpoint metadata does not match the current revised protocol."
    )

    results = payload["results"]

    assert isinstance(results, list)

    return results


# ------------------------------------------------------------
# Load or Start Checkpoint
# ------------------------------------------------------------

if CHECKPOINT_PATH.exists():
    # Resume from the previously saved results when a valid checkpoint
    # is available.
    results = load_checkpoint()

    print(
        f"Existing revised checkpoint loaded: "
        f"{len(results)} completed runs."
    )

else:
    # Start with an empty result list when no checkpoint exists.
    results = []

    print("No revised checkpoint found.")
    print("Starting HPO experiment from the beginning.")


# ------------------------------------------------------------
# Validate Existing Results
# ------------------------------------------------------------
# Represent each completed run using its four experimental identifiers.
# This allows duplicate runs to be detected independently of result order.

completed_runs = {
    (
        result["dataset"],
        result["model"],
        result["training_size"],
        result["subset_seed"],
    )
    for result in results
}

# Every stored result must correspond to a unique experimental run.
assert len(completed_runs) == len(results), (
    "Duplicate run keys detected in checkpoint."
)


# Validate the core protocol identifiers stored in every existing result.
# This provides an additional integrity check beyond checkpoint metadata.

for result in results:
    assert result["protocol_version"] == PROTOCOL_VERSION
    assert result["dataset"] in EXPECTED_DATASETS
    assert result["model"] in MODEL_NAMES
    assert result["training_size"] in HPO_TRAINING_SIZES
    assert result["subset_seed"] in SUBSET_SEEDS

    # Confirm that all random-state settings match the current protocol.
    assert result["model_seed"] == MODEL_SEED
    assert result["cv_seed"] == CV_SEED
    assert result["hpo_seed"] == HPO_SEED


# ------------------------------------------------------------
# Checkpoint Summary
# ------------------------------------------------------------

print()
print(f"Expected runs:          {EXPECTED_RUNS}")
print(f"Completed runs detected:{len(completed_runs)}")
print(f"Checkpoint:              {CHECKPOINT_PATH}")

No revised checkpoint found.
Starting HPO experiment from the beginning.

Expected runs:          270
Completed runs detected:0
Checkpoint:              /kaggle/working/traditional_ml_hpo_revised_protocol_checkpoint.pkl


## Execute Full HPO Experiment

In [14]:
# ============================================================
# Cell 11 - Execute Full HPO Experiment
# ============================================================

print("Executing full Traditional ML + HPO experiment")
print("=" * 60)
print(f"Datasets:        {len(EXPECTED_DATASETS)}")
print(f"HPO sizes:       {len(HPO_TRAINING_SIZES)}")
print(f"Subset seeds:    {len(SUBSET_SEEDS)}")
print(f"Models:          {len(MODEL_NAMES)}")
print(f"Expected runs:   {EXPECTED_RUNS}")
print(f"HPO iterations:  {HPO_N_ITER}")
print(f"CV splits:       {HPO_CV_SPLITS}")
print("=" * 60)

# Track total experiment runtime and resume from completed runs.
experiment_start = time.time()
run_counter = len(completed_runs)

# Iterate over the complete Cartesian product of the protocol-defined
# datasets, HPO training sizes, subset seeds, and models.
for dataset_name in EXPECTED_DATASETS:
    for training_size in HPO_TRAINING_SIZES:
        for subset_seed in SUBSET_SEEDS:
            for model_name in MODEL_NAMES:

                # Unique identifier for one experimental configuration.
                run_key = (
                    dataset_name,
                    model_name,
                    training_size,
                    subset_seed,
                )

                # Skip runs already stored in the checkpoint.
                if run_key in completed_runs:
                    continue

                run_counter += 1

                print()
                print(
                    f"[{run_counter}/{EXPECTED_RUNS}] "
                    f"{dataset_name} | {model_name} | "
                    f"n={training_size} | "
                    f"subset_seed={subset_seed}"
                )

                run_start = time.time()

                # Execute HPO on the predefined training subset
                # and evaluate the refitted best model on the fixed test set.
                result = run_hpo_experiment(
                    dataset_name=dataset_name,
                    model_name=model_name,
                    training_size=training_size,
                    subset_seed=subset_seed,
                )

                run_elapsed = time.time() - run_start

                # ------------------------------------------------
                # Result validation before checkpointing
                # ------------------------------------------------

                # Define the minimum schema required for a valid result.
                required_keys = {
                    "protocol_version",
                    "dataset",
                    "model",
                    "training_size",
                    "subset_seed",
                    "model_seed",
                    "cv_seed",
                    "hpo_seed",
                    "classification_threshold",
                    "best_cv_score",
                    "best_params",
                    "hpo_time",
                    "prediction_time",
                    *ALL_METRICS,
                }

                missing_keys = (
                    required_keys
                    - set(result.keys())
                )

                assert not missing_keys, (
                    f"Missing result keys: {missing_keys}"
                )

                # Verify that the result belongs to the exact
                # protocol configuration that was requested.
                assert result["protocol_version"] == PROTOCOL_VERSION
                assert result["dataset"] == dataset_name
                assert result["model"] == model_name
                assert result["training_size"] == training_size
                assert result["subset_seed"] == subset_seed
                assert result["model_seed"] == MODEL_SEED
                assert result["cv_seed"] == CV_SEED
                assert result["hpo_seed"] == HPO_SEED
                assert (
                    result["classification_threshold"]
                    == CLASSIFICATION_THRESHOLD
                )

                # Ensure all evaluation metrics are finite and
                # within their valid [0, 1] range.
                for metric in ALL_METRICS:
                    value = result[metric]
                    assert np.isfinite(value), (
                        f"Non-finite {metric}: {value}"
                    )
                    assert 0.0 <= value <= 1.0, (
                        f"Invalid {metric}: {value}"
                    )

                # Validate the HPO score, selected parameters,
                # and recorded execution times.
                assert np.isfinite(result["best_cv_score"])
                assert 0.0 <= result["best_cv_score"] <= 1.0
                assert isinstance(result["best_params"], dict)
                assert result["hpo_time"] >= 0
                assert result["prediction_time"] >= 0

                # ------------------------------------------------
                # Append and save checkpoint immediately
                # ------------------------------------------------

                # Only persist a run after all integrity checks pass.
                results.append(result)
                completed_runs.add(run_key)

                # Save after every successful run so an interrupted
                # experiment can resume without repeating completed runs.
                save_checkpoint(results)

                print(
                    f"  ROC-AUC: {result['roc_auc']:.4f} | "
                    f"PR-AUC: {result['pr_auc']:.4f}"
                )
                print(
                    f"  Best CV: {result['best_cv_score']:.4f}"
                )
                print(
                    f"  HPO time: {result['hpo_time']:.2f} s"
                )
                print(
                    f"  Prediction time: {result['prediction_time']:.2f} s"
                )
                print(
                    f"  Run time: {run_elapsed:.2f} s"
                )
                print(
                    f"  Checkpoint saved: "
                    f"{len(results)}/{EXPECTED_RUNS}"
                )


# Measure total wall-clock time for the complete execution,
# including any runs resumed from the existing checkpoint.
elapsed_time = time.time() - experiment_start

print()
print("=" * 60)
print("FULL HPO EXPERIMENT EXECUTION FINISHED")
print("=" * 60)
print(f"Completed runs: {len(results)}")
print(f"Expected runs:  {EXPECTED_RUNS}")
print(f"Elapsed time:   {elapsed_time:.2f} s")

# Final completeness checks ensure that every protocol-defined
# configuration was executed exactly once.
assert len(results) == EXPECTED_RUNS
assert len(completed_runs) == EXPECTED_RUNS

print(f"✓ All {EXPECTED_RUNS} HPO runs completed.")
print(f"✓ Checkpoint: {CHECKPOINT_PATH}")

Executing full Traditional ML + HPO experiment
Datasets:        2
HPO sizes:       9
Subset seeds:    5
Models:          3
Expected runs:   270
HPO iterations:  20
CV splits:       5

[1/270] diabetes | logistic_regression | n=32 | subset_seed=42
  ROC-AUC: 0.6964 | PR-AUC: 0.2865
  Best CV: 0.7389
  HPO time: 0.41 s
  Prediction time: 0.01 s
  Run time: 0.47 s
  Checkpoint saved: 1/270

[2/270] diabetes | random_forest | n=32 | subset_seed=42
  ROC-AUC: 0.7245 | PR-AUC: 0.3014
  Best CV: 0.8444
  HPO time: 12.94 s
  Prediction time: 0.10 s
  Run time: 13.07 s
  Checkpoint saved: 2/270

[3/270] diabetes | xgboost | n=32 | subset_seed=42
  ROC-AUC: 0.6849 | PR-AUC: 0.2590
  Best CV: 0.8222
  HPO time: 3.20 s
  Prediction time: 0.04 s
  Run time: 3.28 s
  Checkpoint saved: 3/270

[4/270] diabetes | logistic_regression | n=32 | subset_seed=123
  ROC-AUC: 0.7857 | PR-AUC: 0.3765
  Best CV: 0.7500
  HPO time: 0.41 s
  Prediction time: 0.01 s
  Run time: 0.47 s
  Checkpoint saved: 4/270

[5/

## Final HPO Result Integrity Check

In [15]:
# ============================================================
# Cell 12 - Final HPO Result Integrity Check
# ============================================================

print("Running final HPO result integrity check...")
print("=" * 60)

# Confirm that the full expected number of HPO runs is available.
assert len(results) == EXPECTED_RUNS

# Convert the list of validated result dictionaries into a tabular form
# for vectorized integrity checks across the complete experiment.
results_df = pd.DataFrame(results)

# ------------------------------------------------------------
# Expected result schema
# ------------------------------------------------------------

# Define the columns that every completed HPO result must contain.
EXPECTED_RESULT_COLUMNS = [
    "protocol_version",
    "dataset",
    "model",
    "training_size",
    "subset_seed",
    "model_seed",
    "cv_seed",
    "hpo_seed",
    "classification_threshold",
    "best_cv_score",
    "best_params",
    "hpo_time",
    "prediction_time",
    *ALL_METRICS,
]

assert set(EXPECTED_RESULT_COLUMNS).issubset(results_df.columns)

# ------------------------------------------------------------
# Experimental key uniqueness
# ------------------------------------------------------------

# These fields uniquely identify one experimental configuration.
key_columns = [
    "dataset",
    "model",
    "training_size",
    "subset_seed",
]

# No configuration may occur more than once in the final results.
assert results_df.duplicated(subset=key_columns).sum() == 0

# ------------------------------------------------------------
# Exact expected combinations
# ------------------------------------------------------------

# Construct the complete set of protocol-defined experiment keys.
expected_keys = {
    (
        dataset_name,
        model_name,
        training_size,
        subset_seed,
    )
    for dataset_name in EXPECTED_DATASETS
    for model_name in MODEL_NAMES
    for training_size in HPO_TRAINING_SIZES
    for subset_seed in SUBSET_SEEDS
}

# Extract the actual experiment keys from the completed results.
actual_keys = set(
    results_df[key_columns].itertuples(index=False, name=None)
)

# Require an exact match: no missing and no unexpected configurations.
assert actual_keys == expected_keys

# ------------------------------------------------------------
# Protocol metadata
# ------------------------------------------------------------

# Verify that all runs used the same protocol version and fixed seeds.
assert results_df["protocol_version"].eq(PROTOCOL_VERSION).all()
assert results_df["model_seed"].eq(MODEL_SEED).all()
assert results_df["cv_seed"].eq(CV_SEED).all()
assert results_df["hpo_seed"].eq(HPO_SEED).all()

# Confirm that every run used the predefined classification threshold.
assert results_df["classification_threshold"].eq(
    CLASSIFICATION_THRESHOLD
).all()

# ------------------------------------------------------------
# Metrics and timing
# ------------------------------------------------------------

# Validate every recorded evaluation metric for completeness,
# numerical validity, and the expected [0, 1] range.
for metric in ALL_METRICS:
    assert results_df[metric].notna().all()
    assert np.isfinite(results_df[metric].astype(float)).all()
    assert (
        (results_df[metric] >= 0)
        & (results_df[metric] <= 1)
    ).all()

# Validate the cross-validation score returned by HPO.
assert results_df["best_cv_score"].notna().all()
assert np.isfinite(results_df["best_cv_score"].astype(float)).all()
assert (
    (results_df["best_cv_score"] >= 0)
    & (results_df["best_cv_score"] <= 1)
).all()

# Execution times must be present and non-negative.
assert results_df["hpo_time"].notna().all()
assert (results_df["hpo_time"] >= 0).all()

assert results_df["prediction_time"].notna().all()
assert (results_df["prediction_time"] >= 0).all()

# Best hyperparameters must remain native dictionaries in the
# serialized PKL results for later inspection and reproducibility.
assert results_df["best_params"].map(
    lambda value: isinstance(value, dict)
).all()

# ------------------------------------------------------------
# Dataset/model group counts
# ------------------------------------------------------------

# Each dataset-model pair must contain every HPO training size
# and every subset seed exactly once.
expected_per_group = (
    len(HPO_TRAINING_SIZES)
    * len(SUBSET_SEEDS)
)

group_counts = (
    results_df
    .groupby(["dataset", "model"])
    .size()
)

assert (group_counts == expected_per_group).all()

# ------------------------------------------------------------
# Final integrity summary
# ------------------------------------------------------------

print(f"✓ Total rows: {len(results_df)}")
print(f"✓ Expected rows: {EXPECTED_RUNS}")
print("✓ No duplicate experimental keys.")
print("✓ All expected dataset/model/size/seed combinations present.")
print("✓ Protocol metadata verified.")
print("✓ Metrics and timing values verified.")
print("✓ Best-parameter dictionaries verified.")
print("✓ Dataset/model group counts verified.")

print()
print("=" * 60)
print("✓ FINAL HPO INTEGRITY CHECK PASSED")
print("=" * 60)

Running final HPO result integrity check...
✓ Total rows: 270
✓ Expected rows: 270
✓ No duplicate experimental keys.
✓ All expected dataset/model/size/seed combinations present.
✓ Protocol metadata verified.
✓ Metrics and timing values verified.
✓ Best-parameter dictionaries verified.
✓ Dataset/model group counts verified.

✓ FINAL HPO INTEGRITY CHECK PASSED


## Save Final HPO Results

In [16]:
# ============================================================
# Cell 13 - Save Final HPO Results — CSV and Pickle
# ============================================================

import json

# Define persistent output paths for the final experiment results.
PKL_PATH = Path(
    "/kaggle/working/"
    "traditional_ml_hpo_results_final.pkl"
)

CSV_PATH = Path(
    "/kaggle/working/"
    "traditional_ml_hpo_results_final.csv"
)

print("Saving final HPO results...")
print("=" * 60)

# ------------------------------------------------------------
# Prepare deterministic result ordering
# ------------------------------------------------------------

# Convert results to a DataFrame and sort by the experimental
# identifiers so that the exported files have a reproducible order.
results_df = (
    pd.DataFrame(results)
    .sort_values(
        by=[
            "dataset",
            "model",
            "training_size",
            "subset_seed",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Save Pickle
# ------------------------------------------------------------

# Preserve native Python objects, including best_params dictionaries,
# for subsequent analysis without converting their data types.
with open(PKL_PATH, "wb") as handle:
    pickle.dump(
        results_df,
        handle,
        protocol=pickle.HIGHEST_PROTOCOL,
    )


# ------------------------------------------------------------
# Save CSV
# ------------------------------------------------------------

# CSV cannot directly represent nested dictionaries, so serialize
# best_params as deterministic JSON strings for portability.
results_csv = results_df.copy()

results_csv["best_params"] = results_csv["best_params"].apply(
    lambda value: json.dumps(value, sort_keys=True)
)

# Export the tabular results without the pandas index.
results_csv.to_csv(
    CSV_PATH,
    index=False,
)


# ------------------------------------------------------------
# Verify saved files
# ------------------------------------------------------------

# Confirm that both output files were created successfully.
assert PKL_PATH.exists()
assert CSV_PATH.exists()

# Reload both files to verify that the serialized outputs are readable.
loaded_pkl = pd.read_pickle(PKL_PATH)
loaded_csv = pd.read_csv(CSV_PATH)

# Confirm that both exports contain the complete experiment.
assert len(loaded_pkl) == EXPECTED_RUNS
assert len(loaded_csv) == EXPECTED_RUNS

# The PKL export should preserve the complete DataFrame schema.
assert list(loaded_pkl.columns) == list(results_df.columns)

print(f"✓ PKL saved: {PKL_PATH}")
print(f"✓ CSV saved: {CSV_PATH}")
print(f"✓ Shape: {results_df.shape}")

print()
print("=" * 60)
print("✓ HPO RESULTS SAVED")
print("=" * 60)

Saving final HPO results...
✓ PKL saved: /kaggle/working/traditional_ml_hpo_results_final.pkl
✓ CSV saved: /kaggle/working/traditional_ml_hpo_results_final.csv
✓ Shape: (270, 20)

✓ HPO RESULTS SAVED


## Final HPO Results Summary

In [17]:
# ============================================================
# Cell 14 - Final HPO Summary
# ============================================================

print("Final Traditional ML + HPO results summary")
print("=" * 60)

# Count completed runs for each dataset-model combination.
counts = (
    results_df
    .groupby(["dataset", "model"])
    .size()
    .unstack(fill_value=0)
)

print("\nRuns per dataset and model:")
display(counts)

# Report the training sizes evaluated by the HPO protocol.
print("\nHPO training sizes:")
print(list(HPO_TRAINING_SIZES))

# Report the predefined subset seeds used for repeated experiments.
print("\nSubset seeds:")
print(list(SUBSET_SEEDS))

# Report the primary metrics used for the main evaluation.
print("\nPrimary metrics:")
print(list(PRIMARY_METRICS))

# Report the secondary metrics retained for supplementary analysis.
print("\nSecondary metrics:")
print(list(SECONDARY_METRICS))

print()
print("=" * 60)
print("✓ FINAL TRADITIONAL ML + HPO RESULTS READY")
print("=" * 60)

# Confirm the final result table contains the complete experiment.
print(f"Total HPO runs: {len(results_df)}")
print(f"Expected HPO runs: {EXPECTED_RUNS}")

# Report the locations of the validated final result artifacts.
print(f"PKL: {PKL_PATH}")
print(f"CSV: {CSV_PATH}")

Final Traditional ML + HPO results summary

Runs per dataset and model:


model,logistic_regression,random_forest,xgboost
dataset,,,
cardiovascular,45,45,45
diabetes,45,45,45



HPO training sizes:
[32, 64, 128, 256, 512, 1024, 2048, 4096, 8192]

Subset seeds:
[42, 123, 456, 789, 1011]

Primary metrics:
['roc_auc', 'pr_auc']

Secondary metrics:
['accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1']

✓ FINAL TRADITIONAL ML + HPO RESULTS READY
Total HPO runs: 270
Expected HPO runs: 270
PKL: /kaggle/working/traditional_ml_hpo_results_final.pkl
CSV: /kaggle/working/traditional_ml_hpo_results_final.csv
